In [4]:
pip install pypdf


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import os
os.environ["PINECONE_API_KEY"] = "pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ"
print("Pinecone API Key:", os.getenv("PINECONE_API_KEY"))
from Search_or_Discuss import classify_and_extract
from data_utils import fetch_papers, verify_results, chunk_papers
from Database import create_index, save_embeddings_to_index, delete_index
from retrieval import answer_query, clear_memory
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))




Pinecone API Key: pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ


In [ ]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

def handleUserQuery(user_input, user_index):
    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)

    if classification["action"] == "search":
        filters = classification['filters']
        print("\nRunning Search with filters:", filters)

        # Step 1: Fetch papers
        papers = fetch_papers(filters)

        # Step 2: Verify & Filter papers
        filtered_papers = verify_results(papers, filters)

        # Step 3: Chunk papers
        paper_chunks = chunk_papers(filtered_papers)

        # Print sample chunks and save embeddings (update as needed)
        # Now, iterate over up to 100 chunks
        for i, chunk in enumerate(paper_chunks[:300]):
            text = chunk["text"]  # Use dictionary indexing
            save_embeddings_to_index(index_name=user_index, text=[text])

    elif classification["action"] == "discuss":
        question = classification['question']
        print("\nDiscussion Topic:", question)
        # Retrieve up to 100 chunks during discussion as well
        answer = answer_query(question, index_obj, top_k=100, use_memory=True)
        print("Generated Answer:")
        print(answer)
    else:
        # Default behavior if classification doesn't match expected actions:
        print("Action not recognized. Please try again.")

# Use a loop to enable continuous conversation
while True:
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        break
    handleUserQuery(user_query, index_name)
delete_index(index_name=index_name)
clear_memory()

Index '2jkdr4k1ot' created successfully!

Running Search with filters: {'query': 'machine learning', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': '20210101', 'end_date': '20211231', 'max_results': '10', 'sort_by': 'newest'}
Extracted 762 chunks from 'A free energy principle for generic quantum systems'
Extracted 653 chunks from 'Studying the Interplay between Information Loss and Operation Loss in Representations for Classification'
Extracted 203 chunks from 'Learning Agent State Online with Recurrent Generate-and-Test'
Extracted 369 chunks from 'Constraint Sampling Reinforcement Learning: Incorporating Expertise For Faster Learning'
Extracted 282 chunks from 'Persformer: A Transformer Architecture for Topological Machine Learning'
Extracted 966 chunks from 'Visual and Object Geo-localization: A Comprehensive Survey'
Extracted 311 chunks from 'Accelerated Primal-Dual Gradient Method for Smooth 